# Pretraining notebook

## Section 1: Generate text before training

Load your current model

Enter a few prompts and note the output.

Try these:
 - love is
 - baby
 - tonight

Generate 20 tokens and note the output.

Reflection: Is there any quality or meaning to the output or is it largely random text?  Why?

The output is largely random and does not have meaningful quality before training. The model was created with randomly initialized weights, so it has not learned English words, grammar, lyric patterns, or which characters usually follow one another. It preserves the original prompt, but the newly generated characters are mostly random choices from the tokenizer’s vocabulary. The mixed-language symbols and characters occur because they are included in the training text vocabulary; after training, the model should generate more consistent text patterns.



In [2]:
import torch

torch.manual_seed(123)

from weird_ai.model import WeirdAIModel
from weird_ai.tokenizer import SimpleCharacterTokenizer
from weird_ai.generation import (
    generate_text_simple,
    text_to_token_ids,
    token_ids_to_text,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

with open("../data/processed/lyrics_sample.txt", "r", encoding="utf-8") as file:
    lyrics_text = file.read()

tokenizer = SimpleCharacterTokenizer(lyrics_text)

model = WeirdAIModel(vocab_size=len(tokenizer.chars))
model = model.to(device)
model.eval()

print("Vocabulary size:", len(tokenizer.chars))
print("Model loaded on:", device)

prompts = ["love is", "baby", "tonight"]

for prompt in prompts:
    token_ids = generate_text_simple(
    model=model,
    input_ids=text_to_token_ids(prompt, tokenizer).to(device),
    max_new_tokens=20,
    context_size=64,
)

    output_text = token_ids_to_text(token_ids, tokenizer)

    print(f"\nPrompt: {prompt}")
    print("Output:", output_text)

Device: cpu
Vocabulary size: 681
Model loaded on: cpu

Prompt: love is
Output: love is®우치ğ你世있想소베위결장оてUâ—Ñ잠

Prompt: baby
Output: baby가Á져久LÓカ離会8想소베很ā춘íフr찾

Prompt: tonight
Output: tonight世爱금디悟陪장:개醒구‘半인てU폼к點ä


## Section 2: Understanding Logits

Using the logits starter code, do the following:
 - Apply softmax
 - Identify the highest probability token

Reflection: Why is softmax necessary before interpreting logits as probabilities?

Softmax is necessary because logits are raw prediction scores, not probabilities. Logits can be negative or positive and do not necessarily add up to 1, so they cannot be interpreted directly as chances. Softmax converts the logits into values between 0 and 1 that sum to 1, making it possible to compare the likelihood of each possible next token. In this example, token 1 had the highest softmax probability, so it was selected as the most likely prediction.

In [3]:
logits = torch.tensor([
    [1.5, 2.0, 0.5]
])

probabilities = torch.softmax(logits, dim=-1)

print(probabilities)
print(probabilities.sum())

predicted_token = torch.argmax(probabilities)

print(predicted_token)

tensor([[0.3315, 0.5465, 0.1220]])
tensor(1.0000)
tensor(1)


## Section 3: Cross Entropy Loss

### Textbook 
The textbook discussion begins on page 136

### Manually calculate
 - Probabilities
 - Log probabilities
 - Average negative log probabilities

### Compare with torch
Compare your manual results to the torch cross entropy results

```python
     torch.nn.functional.cross_entropy(...)
'''


In [4]:
import torch

# Manually calculate values

probs = torch.tensor([
    0.7,
    0.2,
    0.1
])

target_index = 0

In [5]:
target_probability = probs[target_index]

print(target_probability)

tensor(0.7000)


In [6]:
log_probability = torch.log(target_probability)

print(log_probability)

tensor(-0.3567)


In [7]:
loss = -log_probability

print(loss)

tensor(0.3567)


In [8]:
logits = torch.tensor([
    [1.5, 2.0, 0.5]
])

target = torch.tensor([0])

cross_entropy_loss = torch.nn.functional.cross_entropy(logits, target)

print(cross_entropy_loss)

tensor(1.1041)


## Section 4: Perplexity

Compute the perplexity

```python
perplexity = torch.exp(loss)
```

**Note:** A perplexity of 10 means the model is roughly as uncertain as choosing among 10 equally likely next tokens.

In [9]:
loss = torch.tensor(2.5)

perplexity = torch.exp(loss)

print(perplexity)

tensor(12.1825)


## Section 5: Understanding the Training Loop

The training loop performs:

1. Iterate through epochs
2. Iterate through batches
3. Zero gradients
4. Calculate loss
5. Backpropagation
6. Optimizer step
7. Evaluate model
8. Generate sample text

Draw or explain this process in your own words.

When looking at this, the training loop repeats the learning process over several epochs. During those epochs, the model processes the training data one batch at a time. For each batch, the model makes predictions about the next tokens and compares those predictions to the correct answers to calculate the loss. The program clears the old gradients, uses backpropagation to determine what caused the loss, and then the optimizer updates the model’s weights so it can improve. After the model trains on the batches, it is evaluated with validation data that was not used for training. Finally, it generates sample text so I can see whether the output is improving and becoming less random.

# Training vs. Validation

You will:
 - Split lyrics into train/validation
 - Create loaders
 - Compute initial loss

Keep note of your: 
 - Training Loss: 
 - Validation Loss: 

In [ ]:
from weird_ai.dataset import LyricsDataset

train_ratio = 0.9

split_index = int(len(lyrics_text) * train_ratio)

train_text = lyrics_text[:split_index]
val_text = lyrics_text[split_index:]

print("Training characters:", len(train_text))
print("Validation characters:", len(val_text))

Training characters: 12156964
Validation characters: 1350774


In [11]:
from torch.utils.data import DataLoader

block_size = 64
batch_size = 8

train_tokens = tokenizer.encode(train_text)
val_tokens = tokenizer.encode(val_text)

train_dataset = LyricsDataset(
    tokens=train_tokens,
    block_size=block_size,
)

val_dataset = LyricsDataset(
    tokens=val_tokens,
    block_size=block_size,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
)

print(train_loader)
print(val_loader)

In [12]:
from weird_ai.trainer import evaluate_model

eval_iter = 5

initial_train_loss, initial_val_loss = evaluate_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    eval_iter=eval_iter,
)

print(f"Initial training loss: {initial_train_loss:.4f}")
print(f"Initial validation loss: {initial_val_loss:.4f}")

Initial training loss: 6.6619
Initial validation loss: 6.6531


## Section 6: Training

Train for one Epoch
```python
train_model(...)
```

Then record the loss before and after training.  

In [13]:
from torch.utils.data import DataLoader, Subset
from weird_ai.trainer import train_model_simple

num_epochs = 1

max_training_examples = 500
max_validation_examples = 100

small_train_dataset = Subset(
    train_dataset,
    range(min(max_training_examples, len(train_dataset))),
)

small_val_dataset = Subset(
    val_dataset,
    range(min(max_validation_examples, len(val_dataset))),
)

small_train_loader = DataLoader(
    small_train_dataset,
    batch_size=8,
    shuffle=True,
)

small_val_loader = DataLoader(
    small_val_dataset,
    batch_size=8,
    shuffle=False,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=0.01,
)

train_losses, val_losses, tokens_seen = train_model_simple(
    model=model,
    train_loader=small_train_loader,
    val_loader=small_val_loader,
    optimizer=optimizer,
    device=device,
    num_epochs=num_epochs,
    eval_freq=10,
    eval_iter=5,
    start_context="love is",
    tokenizer=tokenizer,
    context_size=block_size,
)

initial_loss = initial_train_loss
final_loss = train_losses[-1]

print(f"Initial loss: {initial_loss:.4f}")
print(f"Final loss: {final_loss:.4f}")

Epoch 1 (Step 000000): Train loss 6.525, Val loss 6.564
Epoch 1 (Step 000010): Train loss 5.383, Val loss 5.449
Epoch 1 (Step 000020): Train loss 4.261, Val loss 4.261
Epoch 1 (Step 000030): Train loss 3.615, Val loss 3.599
Epoch 1 (Step 000040): Train loss 3.308, Val loss 3.289
Epoch 1 (Step 000050): Train loss 3.121, Val loss 3.137
Epoch 1 (Step 000060): Train loss 2.982, Val loss 3.065
love is t t t t t t t t t t t t t t t t t t t t t t t t t
Initial loss: 6.6619
Final loss: 2.9821


In [21]:
from pathlib import Path
from weird_ai.trainer import save_checkpoint

checkpoint_path = Path("../models/lesson-05-pretrained/checkpoint.pt")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

save_checkpoint(
    model=model,
    optimizer=optimizer,
    epoch=num_epochs,
    train_losses=train_losses,
    val_losses=val_losses,
    track_tokens_seen=tokens_seen,
    checkpoint_path=checkpoint_path,
)

print(f"Checkpoint saved successfully: {checkpoint_path}")

Checkpoint saved successfully: ..\models\lesson-05-pretrained\checkpoint.pt


In [22]:
print(f"Initial loss: {initial_loss}")
print(f"Final loss: {final_loss}")

Initial loss: 6.66189603805542
Final loss: 2.9821127891540526


## Evaluation

Did the loss decrease?

Why is decreasing loss important?

Yes. The initial training loss was 6.6619, and the final training loss was 2.9821. The validation loss also decreased from 6.6531 to approximately 3.065.

Decreasing loss is important because loss measures how different the model’s predicted next tokens are from the correct next tokens in the training data. A lower loss means the model is making more accurate predictions and learning patterns from the lyrics. The validation loss also decreased, which suggests the model improved on data that was held back from training.

## Before and After Comparison

Prompt:

love is

Before Training:
__________________

After Training:
__________________

Reflection:

How did the output change?

What evidence do you see that the model learned something from the training data?

Before training, the continuation after “love is” consisted of random-looking mixed characters, including symbols and characters from different writing systems. After training, the continuation became more consistent by using spaces and repeated English letters. However, it was still repetitive and did not yet form meaningful lyrics.

The strongest evidence is that the training loss decreased from 6.6619 to 2.9821 and the validation loss decreased from 6.6531 to about 3.065. The generated text also changed from completely random mixed characters to a more consistent repeated pattern of English characters and spaces. This shows the model began learning character patterns from the training corpus, although it would need more training to produce coherent words and lyrics.



In [24]:
import torch

from weird_ai.model import WeirdAIModel
from weird_ai.trainer import load_checkpoint
from weird_ai.generation import generate_and_print_sample

restored_model = WeirdAIModel(vocab_size=len(tokenizer.chars))
restored_model = restored_model.to(device)

restored_optimizer = torch.optim.AdamW(
    restored_model.parameters(),
    lr=1e-3,
    weight_decay=0.01,
)

checkpoint_path = "../models/lesson-05-pretrained/checkpoint.pt"

metadata = load_checkpoint(
    model=restored_model,
    optimizer=restored_optimizer,
    checkpoint_path=checkpoint_path,
    device=device,
)

print("Restored checkpoint metadata:")
print(metadata)

generate_and_print_sample(
    model=restored_model,
    tokenizer=tokenizer,
    device=device,
    start_context="love is",
    context_size=block_size,
)

Restored checkpoint metadata:
{'epoch': 1, 'train_losses': [6.5248973846435545, 5.382694721221924, 4.261410236358643, 3.6150958061218263, 3.3075603008270265, 3.1207922458648683, 2.9821127891540526], 'val_losses': [6.56352424621582, 5.449232769012451, 4.260767936706543, 3.598880386352539, 3.288577365875244, 3.137318468093872, 3.064501333236694], 'track_tokens_seen': [512, 5632, 10752, 15872, 20992, 26112, 31232]}
love is t t t t t t t t t t t t t t t t t t t t t t t t t
